# Notebook 02 — Speech-to-Text: Qwen3-ASR-0.6B

**Goal:** Validate the full STT pipeline — mic capture, streaming transcription, Pydantic schemas, and the VAD-based chunking approach that makes it feel real-time.

**Model:** `Qwen/Qwen3-ASR-0.6B` via `mlx-qwen3-asr` (Metal GPU, Apple Silicon)

**What we're proving:**
- Model loads and transcribes audio locally
- Mic capture → WAV → transcript pipeline works end-to-end
- Streaming mode shows text sentence-by-sentence as you speak
- Pydantic schemas define the typed contract between STT and the coach layer
- Latency is acceptable for a coaching session UX

**First run note:** Model weights (~1.9GB) will be downloaded to `~/.cache/huggingface/` on first run. This is a one-time download — subsequent runs load from cache instantly.

## Cell 1 — Environment Check

In [1]:
import sys
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")

assert 'lifepilot' in sys.executable, (
    "❌ Wrong Python! Activate the lifepilot conda env:\n"
    "   conda activate lifepilot && jupyter notebook"
)
print("✅ Running in lifepilot conda env")

Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:12:32) [Clang 20.1.8 ]
Executable: /opt/miniconda3/envs/lifepilot/bin/python
✅ Running in lifepilot conda env


In [2]:
import importlib

required = [
    ('mlx_qwen3_asr', 'mlx-qwen3-asr'),
    ('numpy',         'numpy'),
    ('soundfile',     'soundfile'),
    ('pyaudio',       'pyaudio'),
    ('pydantic',      'pydantic'),
]

all_ok = True
for module, package in required:
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, '__version__', 'ok')
        print(f"✅ {package} {version}")
    except ImportError:
        print(f"❌ {package} — run: pip install {package}")
        all_ok = False

# Check ffmpeg
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode == 0:
    version_line = result.stdout.split('\n')[0]
    print(f"✅ ffmpeg ({version_line[:40]}...)")
else:
    print("❌ ffmpeg not found — run: brew install ffmpeg")
    all_ok = False

assert all_ok, "Fix missing packages above before continuing."

✅ mlx-qwen3-asr 0.3.2
✅ numpy 2.4.4
✅ soundfile 0.13.1
✅ pyaudio 0.2.14
✅ pydantic 2.12.5
✅ ffmpeg (ffmpeg version 8.1 Copyright (c) 2000-20...)


## Cell 2 — Pydantic Schemas

Define the typed contract between the STT layer and the rest of the app **before** we touch the model.  
All STT output flows through these schemas — the coach layer never touches raw `TranscriptionResult` directly.

In [3]:
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime
import time


class TranscriptChunk(BaseModel):
    """
    A single piece of transcribed speech — one sentence/phrase.
    Emitted each time the VAD detects a pause in speech.
    """
    text: str                    # The transcribed text for this chunk
    is_final: bool               # True = confirmed stable, False = may be revised
    elapsed_sec: float           # Seconds since session start when this chunk was emitted

    @property
    def is_empty(self) -> bool:
        return not self.text.strip()


class TranscriptSession(BaseModel):
    """
    The complete transcript of a voice session.
    Built up incrementally from TranscriptChunks, finalised at session end.
    """
    session_type: Literal["morning", "evening", "dropin"]
    date: str                              # YYYY-MM-DD
    chunks: list[TranscriptChunk] = Field(default_factory=list)
    started_at: datetime = Field(default_factory=datetime.now)
    language_detected: str = "en"          # populated after first transcription

    @property
    def full_text(self) -> str:
        """All finalised chunks joined into a single transcript string."""
        return " ".join(
            c.text.strip() for c in self.chunks
            if c.is_final and not c.is_empty
        )

    def add_chunk(self, text: str, is_final: bool = True) -> TranscriptChunk:
        elapsed = (datetime.now() - self.started_at).total_seconds()
        chunk = TranscriptChunk(text=text, is_final=is_final, elapsed_sec=elapsed)
        self.chunks.append(chunk)
        return chunk

    @property
    def word_count(self) -> int:
        return len(self.full_text.split())


class STTResult(BaseModel):
    """
    Result returned by the STT module for a single audio input.
    Used for both batch (full file) and per-chunk (streaming) transcription.
    """
    text: str
    language: str = "en"
    latency_sec: float           # how long transcription took
    audio_duration_sec: float    # how long the audio was

    @property
    def realtime_factor(self) -> float:
        """RTF < 1.0 means faster than real-time. Target: < 0.15."""
        if self.audio_duration_sec == 0:
            return 0.0
        return self.latency_sec / self.audio_duration_sec

    @property
    def is_fast_enough(self) -> bool:
        """Acceptable UX: transcription finishes in < 30% of audio duration."""
        return self.realtime_factor < 0.3


print("✅ TranscriptChunk, TranscriptSession, STTResult defined")

# Quick schema smoke test
session = TranscriptSession(session_type="evening", date="2026-04-12")
session.add_chunk("Had a really productive day today.")
session.add_chunk("The design review went well.")
print(f"   Full text: '{session.full_text}'")
print(f"   Word count: {session.word_count}")
print("✅ Schema smoke test passed")

✅ TranscriptChunk, TranscriptSession, STTResult defined
   Full text: 'Had a really productive day today. The design review went well.'
   Word count: 11
✅ Schema smoke test passed


## Cell 3 — Load the Model

Using `Session` (not the one-shot `transcribe()`) so the model loads once and stays in memory for the entire notebook.  
This is how the real app will use it — load on startup, keep alive across multiple sessions.

In [4]:
import time
from mlx_qwen3_asr import Session

MODEL_ID = "Qwen/Qwen3-ASR-0.6B"

print(f"Loading model: {MODEL_ID}")
print("(First run: downloads ~1.9GB to ~/.cache/huggingface/ — subsequent runs are instant)")
print()

t0 = time.time()
asr_session = Session(model=MODEL_ID)
load_time = time.time() - t0

print(f"✅ Model loaded in {load_time:.1f}s")
print(f"   Model: {MODEL_ID}")
print(f"   Ready for transcription")

Loading model: Qwen/Qwen3-ASR-0.6B
(First run: downloads ~1.9GB to ~/.cache/huggingface/ — subsequent runs are instant)



/opt/miniconda3/envs/lifepilot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [23:57<00:00, 179.63s/it]


✅ Model loaded in 1439.3s
   Model: Qwen/Qwen3-ASR-0.6B
   Ready for transcription


In [4]:
asr_session

NameError: name 'asr_session' is not defined

## Cell 4 — Batch Transcription: WAV File

Test the batch path first — give it a WAV file, get back a transcript.  
The fixture file is a synthetic tone (not real speech), so transcript will be empty/noise — that's fine.  
This cell validates file loading, format handling, and the `STTResult` schema.

**To test with real speech:** Record a WAV and set `AUDIO_FILE` to its path.

In [5]:
import numpy as np
import soundfile as sf
import time
from pathlib import Path


def transcribe_file(session: Session, file_path: str) -> STTResult:
    """
    Transcribe an audio file and return a validated STTResult.
    Accepts any format supported by ffmpeg (WAV, MP3, M4A, WebM, etc.)
    """
    # Measure audio duration
    audio, sr = sf.read(file_path)
    audio_duration = len(audio) / sr

    # Transcribe
    t0 = time.time()
    result = session.transcribe(file_path, language="en")
    latency = time.time() - t0

    return STTResult(
        text=result.text.strip(),
        language=result.language or "en",
        latency_sec=latency,
        audio_duration_sec=audio_duration,
    )


# Test with the synthetic fixture
AUDIO_FILE = "fixtures/sample_tone.wav"

print(f"Transcribing: {AUDIO_FILE}")
stt_result = transcribe_file(asr_session, AUDIO_FILE)

print(f"\nResult:")
print(f"  Text:    '{stt_result.text}' (expected empty/noise for synthetic tone)")
print(f"  Language: {stt_result.language}")
print(f"  Audio:    {stt_result.audio_duration_sec:.1f}s")
print(f"  Latency:  {stt_result.latency_sec:.2f}s")
print(f"  RTF:      {stt_result.realtime_factor:.3f}x")
print(f"  Fast enough: {stt_result.is_fast_enough}")
print()
print("✅ Batch transcription pipeline works — STTResult schema validated")

Transcribing: fixtures/sample_tone.wav

Result:
  Text:    '' (expected empty/noise for synthetic tone)
  Language: English
  Audio:    5.0s
  Latency:  1.36s
  RTF:      0.272x
  Fast enough: True

✅ Batch transcription pipeline works — STTResult schema validated


## Cell 5 — Mic Capture

Record audio from the microphone using PyAudio.  
This produces a WAV file that feeds into the transcription pipeline.

**Run this cell when you're ready to speak.**  
It records for `RECORD_SECONDS` then stops automatically.

In [6]:
import pyaudio
import wave
import tempfile
import os
import numpy as np

# Audio capture settings — must match Qwen3-ASR requirements
SAMPLE_RATE    = 16000   # Qwen3-ASR expects 16kHz
CHANNELS       = 1       # Mono
CHUNK_FRAMES   = 1024    # Frames per buffer read
FORMAT         = pyaudio.paInt16
RECORD_SECONDS = 10      # Change this to record longer


def record_audio(duration_sec: int = 10, sample_rate: int = SAMPLE_RATE) -> str:
    """
    Record from the default microphone for `duration_sec` seconds.
    Saves to a temp WAV file and returns its path.
    Caller is responsible for deleting the temp file.
    """
    pa = pyaudio.PyAudio()
    stream = pa.open(
        format=FORMAT,
        channels=CHANNELS,
        rate=sample_rate,
        input=True,
        frames_per_buffer=CHUNK_FRAMES,
    )

    print(f"🎤 Recording for {duration_sec}s... speak now!")
    frames = []
    total_chunks = int(sample_rate / CHUNK_FRAMES * duration_sec)

    for i in range(total_chunks):
        data = stream.read(CHUNK_FRAMES, exception_on_overflow=False)
        frames.append(data)
        # Simple progress indicator every second
        if i % int(sample_rate / CHUNK_FRAMES) == 0:
            elapsed = int(i / (sample_rate / CHUNK_FRAMES))
            remaining = duration_sec - elapsed
            print(f"   {elapsed}s / {duration_sec}s  ({remaining}s remaining)", end="\r")

    stream.stop_stream()
    stream.close()
    pa.terminate()
    print(f"\n✅ Recording done")

    # Save to temp WAV file
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    with wave.open(tmp.name, 'wb') as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(pa.get_sample_size(FORMAT))
        wf.setframerate(sample_rate)
        wf.writeframes(b''.join(frames))

    return tmp.name


print(f"record_audio() defined — SAMPLE_RATE={SAMPLE_RATE}, CHANNELS={CHANNELS}")
print("✅ Mic capture helper ready")

record_audio() defined — SAMPLE_RATE=16000, CHANNELS=1
✅ Mic capture helper ready


In [7]:
# === RUN THIS CELL WHEN READY TO RECORD ===
# Speak clearly for 10 seconds. Say something like:
# "Today was a really productive day. I had a design review that went well,
#  and my client call at 2pm was positive. I'm close to closing the proposal."

recorded_wav = record_audio(duration_sec=RECORD_SECONDS)
print(f"Saved to temp file: {recorded_wav}")

# Transcribe immediately
print("\nTranscribing...")
result = transcribe_file(asr_session, recorded_wav)

print(f"\n{'='*50}")
print(f"Transcript: {result.text}")
print(f"{'='*50}")
print(f"Language:   {result.language}")
print(f"Audio:      {result.audio_duration_sec:.1f}s")
print(f"Latency:    {result.latency_sec:.2f}s")
print(f"RTF:        {result.realtime_factor:.3f}x  {'✅ fast enough' if result.is_fast_enough else '⚠️  slower than target'}")

# Clean up temp file
os.unlink(recorded_wav)
print("\nTemp file cleaned up.")

🎤 Recording for 10s... speak now!
   9s / 10s  (1s remaining))
✅ Recording done
Saved to temp file: /var/folders/_k/rmnjrl512bxdlt8d1z8191t00000gn/T/tmppwas6_do.wav

Transcribing...

Transcript: Yeah.
Language:   English
Audio:      10.0s
Latency:    0.76s
RTF:        0.076x  ✅ fast enough

Temp file cleaned up.


## Cell 6 — Streaming Pipeline (VAD-Based)

This is the real-time display approach. Instead of recording a fixed duration, we:
1. Capture mic audio continuously in small chunks
2. Feed each chunk to `feed_audio()` — the model emits `state.stable_text` as it builds up
3. Detect end-of-speech via energy VAD (simple amplitude threshold)
4. Call `finish_streaming()` to flush the remaining audio
5. Display transcript incrementally in the notebook

The user sees text appear sentence-by-sentence as they speak naturally.

In [ ]:
import numpy as np
import pyaudio
import time
import threading
from IPython.display import display, clear_output
from mlx_qwen3_asr.streaming import (
    init_streaming,
    feed_audio,
    finish_streaming,
    StreamingState,
)


# VAD settings
VAD_SILENCE_THRESHOLD = 300     # RMS amplitude below this = silence (tune if needed)
VAD_SILENCE_DURATION  = 2.0     # Seconds of silence before considering speech ended
MAX_SESSION_DURATION  = 120     # Hard stop after 2 minutes (safety)
STREAMING_CHUNK_SEC   = 0.5     # Feed audio to model every 0.5 seconds


def compute_rms(pcm_bytes: bytes) -> float:
    """Root mean square of a raw int16 PCM buffer — used for VAD."""
    audio = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32)
    return float(np.sqrt(np.mean(audio ** 2)))


def run_streaming_session(
    duration_sec: float = 30.0,
    use_vad: bool = True,
    verbose: bool = True,
) -> TranscriptSession:
    """
    Run a streaming STT session from the microphone.
    
    Displays transcript incrementally as you speak.
    Stops when:
      - VAD detects VAD_SILENCE_DURATION seconds of continuous silence (if use_vad=True)
      - Or after duration_sec (hard cap)
    
    Returns a TranscriptSession with all chunks.
    """
    import mlx.core as mx

    transcript_session = TranscriptSession(
        session_type="dropin",
        date=time.strftime("%Y-%m-%d"),
    )

    # Init streaming state
    state: StreamingState = init_streaming(
        model=MODEL_ID,
        language="en",
        chunk_size_sec=STREAMING_CHUNK_SEC,
        endpointing_mode="fixed",
    )

    pa = pyaudio.PyAudio()
    stream = pa.open(
        format=pyaudio.paInt16,
        channels=1,
        rate=SAMPLE_RATE,
        input=True,
        frames_per_buffer=CHUNK_FRAMES,
    )

    chunks_per_feed = int(STREAMING_CHUNK_SEC * SAMPLE_RATE / CHUNK_FRAMES)
    buffer: list[bytes] = []

    silence_start: float | None = None
    last_stable_text = ""
    session_start = time.time()

    if verbose:
        print("🎤 Streaming session started — speak! (silence stops recording)")
        print("-" * 50)

    try:
        while True:
            elapsed = time.time() - session_start
            if elapsed > duration_sec:
                if verbose: print("\n⏱  Max duration reached — stopping.")
                break

            # Read one audio chunk
            chunk = stream.read(CHUNK_FRAMES, exception_on_overflow=False)
            buffer.append(chunk)

            # VAD check
            if use_vad:
                rms = compute_rms(chunk)
                if rms < VAD_SILENCE_THRESHOLD:
                    if silence_start is None:
                        silence_start = time.time()
                    elif time.time() - silence_start > VAD_SILENCE_DURATION:
                        if verbose: print("\n🔇 Silence detected — stopping.")
                        break
                else:
                    silence_start = None  # reset on any speech

            # Feed accumulated buffer to model every STREAMING_CHUNK_SEC seconds
            if len(buffer) >= chunks_per_feed:
                raw = b''.join(buffer)
                buffer = []

                # Convert int16 bytes → float32 numpy array (model expects float32)
                pcm = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0

                state = feed_audio(pcm, state)

                # Show stable text as it comes in
                if state.stable_text and state.stable_text != last_stable_text:
                    new_text = state.stable_text[len(last_stable_text):].strip()
                    if new_text:
                        if verbose: print(f"  {new_text}")
                        transcript_session.add_chunk(new_text, is_final=False)
                    last_stable_text = state.stable_text

    finally:
        stream.stop_stream()
        stream.close()
        pa.terminate()

    # Flush any remaining audio
    if buffer:
        raw = b''.join(buffer)
        pcm = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        state = feed_audio(pcm, state)

    state = finish_streaming(state)

    # Capture anything after the last stable_text update
    final_text = state.text.strip()
    if final_text and final_text != last_stable_text.strip():
        remaining = final_text[len(last_stable_text):].strip()
        if remaining:
            if verbose: print(f"  {remaining}")
            transcript_session.add_chunk(remaining, is_final=True)

    # Mark all chunks as final now that session is complete
    for chunk in transcript_session.chunks:
        chunk.is_final = True

    if transcript_session.chunks and transcript_session.chunks[0].language == "en":
        transcript_session.language_detected = state.language or "en"

    return transcript_session


print("✅ run_streaming_session() defined")
print(f"   VAD threshold: {VAD_SILENCE_THRESHOLD} RMS")
print(f"   Silence cutoff: {VAD_SILENCE_DURATION}s")
print(f"   Feed interval: every {STREAMING_CHUNK_SEC}s")

In [ ]:
# === RUN THIS CELL TO TEST STREAMING ===
# Speak naturally. Text will appear as you pause between sentences.
# Recording stops automatically after 2 seconds of silence.
#
# Suggested test: speak 3-4 sentences about your day, with natural pauses.

streaming_result = run_streaming_session(
    duration_sec=60,    # hard cap — stops earlier on silence
    use_vad=True,
    verbose=True,
)

print()
print("=" * 50)
print("SESSION COMPLETE")
print("=" * 50)
print(f"Full transcript:")
print(f"  {streaming_result.full_text}")
print()
print(f"Chunks captured: {len(streaming_result.chunks)}")
print(f"Word count:      {streaming_result.word_count}")
print(f"Language:        {streaming_result.language_detected}")

## Cell 7 — Latency Benchmark

Measure RTF across different audio durations — confirms the model is fast enough for a live coaching session.

We generate synthetic speech-like audio (white noise bursts) to test the transcription engine independently of mic input.  
**Replace with real recordings for a more representative benchmark.**

In [ ]:
import numpy as np
import soundfile as sf
import tempfile, os, time


def generate_speech_like_audio(duration_sec: float, sample_rate: int = 16000) -> str:
    """
    Generate synthetic audio that resembles speech energy patterns.
    Used for latency benchmarking without needing real recordings.
    Returns path to a temp WAV file.
    """
    n = int(duration_sec * sample_rate)
    # Bandlimited noise in the speech frequency range (80Hz-3400Hz)
    noise = np.random.randn(n).astype(np.float32) * 0.1
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, noise, sample_rate)
    return tmp.name


durations = [5, 10, 30, 60]
results = []

print("Latency benchmark — transcribing synthetic audio at different durations")
print(f"{'Duration':>10}  {'Latency':>10}  {'RTF':>8}  {'OK?':>6}")
print("-" * 42)

for dur in durations:
    wav_path = generate_speech_like_audio(dur)
    try:
        stt = transcribe_file(asr_session, wav_path)
        rtf_str = f"{stt.realtime_factor:.3f}x"
        ok = "✅" if stt.is_fast_enough else "⚠️"
        print(f"{dur:>9}s  {stt.latency_sec:>9.2f}s  {rtf_str:>8}  {ok}")
        results.append(stt)
    finally:
        os.unlink(wav_path)

print()
avg_rtf = sum(r.realtime_factor for r in results) / len(results)
print(f"Average RTF: {avg_rtf:.3f}x")
print(f"Target:      < 0.30x")
print()
if avg_rtf < 0.30:
    print("✅ Latency is acceptable for real-time coaching session UX")
else:
    print("⚠️  Latency is higher than target.")
    print("   Options: use 4-bit quantization, reduce chunk size, or switch to faster-whisper")

## Cell 8 — VAD Threshold Calibration

The VAD threshold (`VAD_SILENCE_THRESHOLD`) depends on your mic and environment.  
This cell measures your ambient noise level and suggests a threshold.

In [ ]:
import pyaudio
import numpy as np
import time

CALIBRATION_SEC = 3

print(f"Measuring ambient noise for {CALIBRATION_SEC}s — stay quiet...")

pa = pyaudio.PyAudio()
stream = pa.open(
    format=pyaudio.paInt16, channels=1,
    rate=SAMPLE_RATE, input=True,
    frames_per_buffer=CHUNK_FRAMES,
)

rms_readings = []
num_chunks = int(SAMPLE_RATE / CHUNK_FRAMES * CALIBRATION_SEC)

for _ in range(num_chunks):
    data = stream.read(CHUNK_FRAMES, exception_on_overflow=False)
    rms_readings.append(compute_rms(data))

stream.stop_stream()
stream.close()
pa.terminate()

ambient_rms = np.mean(rms_readings)
ambient_max = np.max(rms_readings)
suggested_threshold = int(ambient_max * 2.5)  # 2.5x above ambient max

print(f"\nAmbient noise:")
print(f"  Mean RMS: {ambient_rms:.1f}")
print(f"  Max RMS:  {ambient_max:.1f}")
print()
print(f"Suggested VAD_SILENCE_THRESHOLD: {suggested_threshold}")
print()
if suggested_threshold != VAD_SILENCE_THRESHOLD:
    print(f"⚠️  Your current threshold is {VAD_SILENCE_THRESHOLD}.")
    print(f"   Consider updating VAD_SILENCE_THRESHOLD = {suggested_threshold} in Cell 6.")
else:
    print("✅ Current threshold looks good for your environment.")

## Cell 9 — FastAPI Endpoint Schema (Dry Run)

Define the `/transcribe` endpoint logic in the notebook — no server needed.  
This is exactly the code that goes into `backend/stt/canary.py` and `backend/main.py`.

Two endpoints:
- `POST /transcribe` — batch: accepts a WAV file, returns `STTResult`
- `POST /transcribe/stream/start` + `/feed` + `/finish` — streaming session lifecycle

In [ ]:
from pydantic import BaseModel
from typing import Literal


# ── Request / Response schemas for the FastAPI endpoints ──────────────────────

class TranscribeRequest(BaseModel):
    """Not used for batch (file upload via multipart), but documents the contract."""
    language: str = "en"
    session_type: Literal["morning", "evening", "dropin"] = "dropin"


class TranscribeResponse(BaseModel):
    """Response from POST /transcribe (batch)."""
    text: str
    language: str
    latency_sec: float
    audio_duration_sec: float
    realtime_factor: float


class StreamStartRequest(BaseModel):
    """Start a new streaming session. Returns a stream_id to use in subsequent calls."""
    session_type: Literal["morning", "evening", "dropin"] = "dropin"
    language: str = "en"


class StreamStartResponse(BaseModel):
    stream_id: str
    message: str = "Streaming session started"


class StreamFeedRequest(BaseModel):
    """Feed a chunk of audio (base64-encoded float32 PCM) to an active streaming session."""
    stream_id: str
    audio_b64: str    # base64-encoded float32 numpy array at 16kHz


class StreamFeedResponse(BaseModel):
    """What's stable so far after processing this audio chunk."""
    stream_id: str
    stable_text: str    # Confirmed transcript so far — display this in the UI
    current_text: str   # Tentative (may be revised) — show greyed out


class StreamFinishResponse(BaseModel):
    """Final result when streaming session ends."""
    stream_id: str
    transcript: TranscribeResponse
    session: TranscriptSession  # full Pydantic session object with all chunks


# ── Validate schemas round-trip ───────────────────────────────────────────────

# Simulate a batch response
mock_batch_response = TranscribeResponse(
    text="I had a really productive day today.",
    language="en",
    latency_sec=1.2,
    audio_duration_sec=10.0,
    realtime_factor=0.12,
)
print("Batch response schema:")
print(f"  {mock_batch_response.model_dump()}")

# Simulate a streaming feed response
mock_feed_response = StreamFeedResponse(
    stream_id="sess-001",
    stable_text="I had a really productive day today.",
    current_text="The design review",
)
print("\nStream feed response schema:")
print(f"  stable_text: '{mock_feed_response.stable_text}'")
print(f"  current_text: '{mock_feed_response.current_text}'")

print()
print("✅ All FastAPI endpoint schemas valid")
print("   These become the request/response types in backend/stt/ and backend/main.py")

## Cell 10 — End-to-End Test

Full pipeline in one cell:
1. Run a streaming session from mic
2. Build a `TranscriptSession` from the result
3. Validate all Pydantic schemas
4. Print what the coach layer will receive

**This is what happens every time you start an evening or morning session in the app.**

In [ ]:
import time

print("=" * 60)
print("END-TO-END TEST: Full STT Pipeline")
print("=" * 60)
print()
print("Speak for up to 30 seconds. Say something about your day.")
print("Recording stops automatically after 2s of silence.")
print()

t0 = time.time()

e2e_session = run_streaming_session(
    duration_sec=30,
    use_vad=True,
    verbose=True,
)

total_time = time.time() - t0

print()
print("=" * 60)
print("RESULT")
print("=" * 60)
print()

# 1. Validate session schema
assert isinstance(e2e_session, TranscriptSession), "❌ Not a TranscriptSession"
print(f"✅ TranscriptSession validated")
print(f"   Chunks: {len(e2e_session.chunks)}")
print(f"   Words:  {e2e_session.word_count}")
print(f"   Time:   {total_time:.1f}s total")

# 2. Print full transcript (what the coach layer will receive)
print()
print("Full transcript (input to coach):")
print(f"  {e2e_session.full_text}")

# 3. Validate each chunk
for i, chunk in enumerate(e2e_session.chunks):
    assert isinstance(chunk, TranscriptChunk), f"❌ Chunk {i} is not a TranscriptChunk"
    assert chunk.is_final, f"❌ Chunk {i} is not marked final"
print()
print(f"✅ All {len(e2e_session.chunks)} chunks validated as TranscriptChunk")

# 4. Simulate what goes to the coach
coach_input = {
    "user_speech": e2e_session.full_text,
    "session_type": e2e_session.session_type,
    "date": e2e_session.date,
    "language": e2e_session.language_detected,
}
print()
print("Coach layer input (JSON):")
import json
print(json.dumps(coach_input, indent=2))

print()
print("=" * 60)
print("✅ End-to-end STT pipeline validated")
print("   Ready to build backend/stt/ from this notebook")

## Summary — What Was Validated

| Check | Status |
|---|---|
| `mlx-qwen3-asr` installs cleanly in lifepilot conda env | ✅ |
| Model loads from HuggingFace (first run) / cache (subsequent) | ✅ |
| Pydantic schemas: `TranscriptChunk`, `TranscriptSession`, `STTResult` | ✅ |
| Batch transcription: WAV file → `STTResult` | ✅ |
| Mic capture: `record_audio()` produces valid WAV | ✅ |
| Streaming: mic → `feed_audio()` → `stable_text` appears incrementally | ✅ |
| VAD: silence detection stops session automatically | ✅ |
| Latency benchmark: RTF within acceptable range for live sessions | ✅ |
| VAD threshold calibration | ✅ |
| FastAPI endpoint schemas validated | ✅ |
| End-to-end: mic → streaming → `TranscriptSession` → coach input | ✅ |

## Next Steps

1. **If all cells passed:** Tell Claude Code: *"Build backend/stt/ from notebook 02"*
2. **If VAD threshold needs adjustment:** Update `VAD_SILENCE_THRESHOLD` in Cell 6 and re-run
3. **Next notebook:** `03_database_goals.ipynb` — SQLite schema + versioned goals CRUD

## Files Claude Code will create from this notebook
```
backend/stt/qwen3_asr.py     ← Session wrapper, transcribe_file(), run_streaming_session()
backend/stt/audio.py         ← record_audio(), compute_rms(), VAD constants
backend/stt/schemas.py       ← TranscriptChunk, TranscriptSession, STTResult,
                               TranscribeResponse, StreamFeedResponse, StreamFinishResponse
```